# 🌲 MÓDULO 3: Entrenamiento de Modelos
## Árbol de Decisión vs Random Forest

**Objetivo:** Entrenar ambos modelos y comparar su rendimiento.

---

## 3.1 Configuración y Preparación de Datos

In [1]:
import sys
import time

EN_COLAB = 'google.colab' in sys.modules

if EN_COLAB:
    print("📍 Ejecutando en Google Colab")
    RUTA_DATOS = '/content/drive/MyDrive/ML_Vuelos/data/'
else:
    print("🐳 Ejecutando en Docker")
    RUTA_DATOS = '../data/'

from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Vuelos_Modulo3") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

🐳 Ejecutando en Docker


In [2]:
# Cargar y preparar datos (repetimos del Módulo 2)
from pyspark.ml.feature import StringIndexer, VectorAssembler

df = spark.read.csv(RUTA_DATOS + "flights_2015_full.csv", header=True, inferSchema=True)

# Indexar categóricas
for col_name in ["AIRLINE", "ORIGIN", "DEST"]:
    indexer = StringIndexer(inputCol=col_name, outputCol=f"{col_name}_idx", handleInvalid="keep")
    df = indexer.fit(df).transform(df)

# Crear vector de features
feature_cols = ["MONTH", "DAY_OF_WEEK", "DEP_HOUR", "DISTANCE", "AIRLINE_idx", "ORIGIN_idx", "DEST_idx"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_ml = assembler.transform(df).select("features", "DEP_DEL15")

# Dividir
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)
print(f"✅ Datos preparados: {train_df.count():,} train / {test_df.count():,} test")

✅ Datos preparados: 4,267,045 train / 1,065,869 test


## 3.2 Entrenar Árbol de Decisión 🌳

In [3]:
#valanceo del modelo
retrasados = train_df.filter(train_df.DEP_DEL15 == 1)
a_tiempo = train_df.filter(train_df.DEP_DEL15 == 0)

# Calculamos la proporción
fraccion = retrasados.count() / a_tiempo.count()

# Muestreamos la clase mayoritaria
a_tiempo_sub = a_tiempo.sample(withReplacement=False, fraction=fraccion, seed=42)

# Unimos para crear el set de entrenamiento balanceado
train_df_balanced = retrasados.union(a_tiempo_sub)

In [4]:
from pyspark.ml.classification import DecisionTreeClassifier

print("🌳 Entrenando Árbol de Decisión...")
inicio = time.time()

dt = DecisionTreeClassifier(
    labelCol="DEP_DEL15",
    featuresCol="features",
    maxDepth=8,          # Profundidad máxima
    minInstancesPerNode=100,  # Mínimo de ejemplos por nodo
    maxBins=330
)

modelo_arbol = dt.fit(train_df_balanced)
tiempo_arbol = time.time() - inicio

print(f"✅ Árbol entrenado en {tiempo_arbol:.2f} segundos")
print(f"   Profundidad: {modelo_arbol.depth}")
print(f"   Nodos: {modelo_arbol.numNodes}")

🌳 Entrenando Árbol de Decisión...
✅ Árbol entrenado en 29.97 segundos
   Profundidad: 8
   Nodos: 233


In [5]:
# Feature Importance del Árbol
print("\n📊 IMPORTANCIA DE VARIABLES (Árbol)")
print("="*40)
importancias = modelo_arbol.featureImportances.toArray()
for nombre, imp in sorted(zip(feature_cols, importancias), key=lambda x: -x[1]):
    barra = "█" * int(imp * 50)
    print(f"{nombre:15} {imp:.4f} {barra}")


📊 IMPORTANCIA DE VARIABLES (Árbol)
DEP_HOUR        0.6737 █████████████████████████████████
AIRLINE_idx     0.1347 ██████
ORIGIN_idx      0.0841 ████
MONTH           0.0566 ██
DEST_idx        0.0395 █
DISTANCE        0.0071 
DAY_OF_WEEK     0.0043 


In [6]:
# Ver estructura del árbol
print("\n🌳 ESTRUCTURA DEL ÁRBOL (primeras líneas):")
print(modelo_arbol.toDebugString[:2000])


🌳 ESTRUCTURA DEL ÁRBOL (primeras líneas):
DecisionTreeClassificationModel: uid=DecisionTreeClassifier_86e1e496f7c0, depth=8, numNodes=233, numClasses=2, numFeatures=7
  If (feature 2 <= 10.5)
   If (feature 2 <= 7.5)
    If (feature 5 in {0.0,3.0,5.0,6.0,8.0,9.0,10.0,11.0,12.0,13.0,14.0,15.0,16.0,17.0,18.0,19.0,20.0,21.0,22.0,23.0,24.0,25.0,26.0,27.0,28.0,29.0,30.0,31.0,32.0,33.0,34.0,35.0,36.0,37.0,38.0,39.0,40.0,41.0,42.0,43.0,44.0,45.0,46.0,47.0,48.0,49.0,50.0,51.0,52.0,53.0,54.0,55.0,56.0,57.0,58.0,59.0,60.0,61.0,62.0,63.0,64.0,65.0,66.0,67.0,68.0,69.0,70.0,71.0,72.0,73.0,74.0,75.0,76.0,77.0,78.0,79.0,80.0,81.0,82.0,83.0,84.0,85.0,86.0,87.0,88.0,89.0,90.0,91.0,92.0,93.0,94.0,95.0,96.0,97.0,98.0,99.0,100.0,101.0,102.0,103.0,104.0,105.0,106.0,107.0,108.0,109.0,110.0,111.0,112.0,113.0,114.0,115.0,116.0,117.0,118.0,119.0,120.0,121.0,122.0,123.0,124.0,125.0,126.0,127.0,128.0,129.0,130.0,131.0,132.0,133.0,134.0,135.0,136.0,137.0,138.0,139.0,140.0,141.0,142.0,143.0,144.0,145.0,146.0,147.

## 3.3 Entrenar Random Forest 🌲🌲🌲

In [7]:
from pyspark.ml.classification import RandomForestClassifier

print("🌲 Entrenando Random Forest (100 árboles)...")
inicio = time.time()

rf = RandomForestClassifier(
    labelCol="DEP_DEL15",
    featuresCol="features",
    numTrees=80,        # Número de árboles
    maxDepth=8,          # Profundidad máxima por árbol
    seed=42,
    maxBins=330
)

modelo_bosque = rf.fit(train_df_balanced)
tiempo_bosque = time.time() - inicio

print(f"✅ Bosque entrenado en {tiempo_bosque:.2f} segundos")
print(f"   Árboles: {modelo_bosque.numTrees}")

🌲 Entrenando Random Forest (100 árboles)...
✅ Bosque entrenado en 94.95 segundos
   Árboles: RandomForestClassifier_ae12da14981d__numTrees


In [8]:
# Feature Importance del Bosque
print("\n📊 IMPORTANCIA DE VARIABLES (Random Forest)")
print("="*40)
importancias_rf = modelo_bosque.featureImportances.toArray()
for nombre, imp in sorted(zip(feature_cols, importancias_rf), key=lambda x: -x[1]):
    barra = "█" * int(imp * 50)
    print(f"{nombre:15} {imp:.4f} {barra}")


📊 IMPORTANCIA DE VARIABLES (Random Forest)
DEP_HOUR        0.6829 ██████████████████████████████████
AIRLINE_idx     0.1135 █████
ORIGIN_idx      0.0824 ████
MONTH           0.0612 ███
DEST_idx        0.0356 █
DISTANCE        0.0198 
DAY_OF_WEEK     0.0046 


## 3.4 Comparación de Tiempos

In [9]:
print("\n⏱️ COMPARACIÓN DE TIEMPOS DE ENTRENAMIENTO")
print("="*45)
print(f"{'Modelo':<25} {'Tiempo':>10} {'Factor':>8}")
print("-"*45)
print(f"{'Árbol de Decisión':<25} {tiempo_arbol:>8.2f}s {1.0:>7.1f}x")
print(f"{'Random Forest (100)':<25} {tiempo_bosque:>8.2f}s {tiempo_bosque/tiempo_arbol:>7.1f}x")
print("="*45)


⏱️ COMPARACIÓN DE TIEMPOS DE ENTRENAMIENTO
Modelo                        Tiempo   Factor
---------------------------------------------
Árbol de Decisión            29.97s     1.0x
Random Forest (100)          94.95s     3.2x


---
## ✅ CHECKPOINT MÓDULO 3

Ahora tienes:

| Modelo | Variable | Tiempo |
|--------|----------|--------|
| Árbol de Decisión | `modelo_arbol` | Rápido |
| Random Forest | `modelo_bosque` | Más lento |

**Siguiente:** → `Modulo4_Evaluacion.ipynb`

diccionario indexacion del origen

In [12]:
# Función para recuperar las etiquetas
def recuperar_etiquetas(df, nombre_columna_idx):
    meta = df.schema[nombre_columna_idx].metadata
    return meta['ml_attr']['vals']

dicc_airline = recuperar_etiquetas(df, "AIRLINE_idx")
dicc_origin  = recuperar_etiquetas(df, "ORIGIN_idx")
dicc_dest    = recuperar_etiquetas(df, "DEST_idx")

# Imprimir TODAS las aerolíneas
print("✈️ TODAS LAS AEROLÍNEAS:")
for i, nombre in enumerate(dicc_airline):
    print(f"{float(i)}: {nombre}")

# Imprimir TODOS los orígenes (¡Aquí puede haber más de 300!)
print("\n🏙️ TODOS LOS ORÍGENES:")
for i, nombre in enumerate(dicc_origin):
    print(f"{float(i)}: {nombre}")

✈️ TODAS LAS AEROLÍNEAS:
0.0: WN
1.0: DL
2.0: AA
3.0: OO
4.0: EV
5.0: UA
6.0: MQ
7.0: B6
8.0: US
9.0: AS
10.0: NK
11.0: F9
12.0: HA
13.0: VX
14.0: __unknown

🏙️ TODOS LOS ORÍGENES:
0.0: ATL
1.0: ORD
2.0: DFW
3.0: DEN
4.0: LAX
5.0: SFO
6.0: PHX
7.0: IAH
8.0: LAS
9.0: MSP
10.0: MCO
11.0: SEA
12.0: DTW
13.0: BOS
14.0: EWR
15.0: CLT
16.0: LGA
17.0: SLC
18.0: JFK
19.0: BWI
20.0: MDW
21.0: DCA
22.0: FLL
23.0: SAN
24.0: MIA
25.0: PHL
26.0: TPA
27.0: DAL
28.0: HOU
29.0: BNA
30.0: PDX
31.0: STL
32.0: HNL
33.0: OAK
34.0: AUS
35.0: MSY
36.0: MCI
37.0: SJC
38.0: SMF
39.0: SNA
40.0: CLE
41.0: IAD
42.0: RDU
43.0: MKE
44.0: SAT
45.0: RSW
46.0: IND
47.0: SJU
48.0: CMH
49.0: PIT
50.0: PBI
51.0: OGG
52.0: CVG
53.0: ABQ
54.0: BUR
55.0: BDL
56.0: JAX
57.0: ONT
58.0: BUF
59.0: OMA
60.0: OKC
61.0: ANC
62.0: RIC
63.0: TUS
64.0: MEM
65.0: TUL
66.0: RNO
67.0: BHM
68.0: ELP
69.0: CHS
70.0: BOI
71.0: KOA
72.0: PVD
73.0: GRR
74.0: LIH
75.0: LIT
76.0: SDF
77.0: GEG
78.0: ORF
79.0: XNA
80.0: MSN
81.0: PSP
82.0: LGB